# Chapter 11 — Capstone: simulate a dataset, then learn it back

**Goal:** use everything at once. You will
1. **simulate** a Titanic-like dataset from survival probabilities *you* choose (NumPy + pandas),
2. run the **same pipeline as MLMA Lab 1**: missing values, one-hot encoding, train/test split, decision tree, accuracy (scikit-learn),
3. compare the model with the **truth**, which you know exactly because you made the data. With real data you never get to do this.

Exercises in this chapter build on each other, so do them in order. If one fails, fix it before moving on.

### scikit-learn in 30 seconds

Every scikit-learn object follows the pattern you built in Chapter 5:

```python
model = DecisionTreeClassifier(max_depth=3)   # settings
model.fit(X_train, y_train)                   # learn
y_pred = model.predict(X_test)                # use
proba = model.predict_proba(X_test)           # class probabilities, shape (n, 2)

encoder = OneHotEncoder(drop="first", sparse_output=False)
encoder.fit(train_df[["Sex"]])                # learn the categories (training data only!)
encoder.transform(test_df[["Sex"]])           # apply them
```

`X` is a 2-D table (rows = samples, columns = features) and `y` is a 1-D array of labels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(2026)
N = 3000

# The "true world": survival probability by sex and class (index 0 is unused so that the class number is the index)
P_SURVIVE = {"female": np.array([np.nan, 0.95, 0.85, 0.50]),
             "male":   np.array([np.nan, 0.40, 0.15, 0.12])}
CHILD_BONUS = 0.25   # children under 12 get +0.25 (capped at 1)

---
### Exercise 11.1: Simulate the features
Create a DataFrame `people` with `N` rows and these columns, drawing in this order from `rng`:
- `Pclass`: `rng.choice([1, 2, 3], size=N, p=[0.25, 0.20, 0.55])`
- `Sex`: `"female"` or `"male"` with probabilities `[0.35, 0.65]`
- `Age`: normal with mean 30 and std 14, then `np.clip(..., 0.5, 80)` and rounded to 1 decimal (`np.round(..., 1)`)
- `Embarked`: `"S"`, `"C"`, `"Q"` with probabilities `[0.72, 0.19, 0.09]`

In [ ]:
people = ...
people.head()

In [ ]:
assert people.shape == (N, 4) and list(people.columns) == ["Pclass", "Sex", "Age", "Embarked"]
assert set(people["Pclass"]) == {1, 2, 3} and set(people["Sex"]) == {"female", "male"}
assert people["Age"].between(0.5, 80).all()
assert abs((people["Sex"] == "female").mean() - 0.35) < 0.03
print("Exercise 11.1 passed")

### Exercise 11.2: The true survival probability of each person
Add a column `p_true` holding each person's survival probability:
1. look up the base probability from `P_SURVIVE` using their sex and class.
   For one sex at a time: `P_SURVIVE["female"][people["Pclass"]]` is fancy indexing (Chapter 6).
   Combine the two sexes with `np.where(people["Sex"] == "female", ..., ...)`.
2. add `CHILD_BONUS` where `Age < 12`
3. cap at 1 with `np.minimum(..., 1.0)`

In [ ]:
people["p_true"] = ...

In [ ]:
row = people.iloc[0]
expected = min(P_SURVIVE[row["Sex"]][row["Pclass"]] + (CHILD_BONUS if row["Age"] < 12 else 0), 1)
assert np.isclose(row["p_true"], expected)
assert people["p_true"].between(0, 1).all() and people["p_true"].notna().all()
print("Exercise 11.2 passed")

### Exercise 11.3: Flip the coins
Each person survives with probability `p_true`: draw one uniform number per person and compare.
Store the result as integers 0/1 in a column `Survived`.

**Predict first:** roughly what overall survival rate do you expect? Write it in `my_guess`.
Then compute the expected rate exactly as the mean of `p_true`, and set `observed_rate` to the mean of `Survived`.

In [ ]:
my_guess = ...
observed_rate = ...
print(f"guess {my_guess}, expected {people['p_true'].mean():.3f}, observed {observed_rate:.3f}")

In [ ]:
assert set(people["Survived"].unique()) == {0, 1}
assert abs(observed_rate - people["p_true"].mean()) < 0.03
print("Exercise 11.3 passed")

### Exercise 11.4: Make it messy
Real data has holes. Make a copy `data = people.drop(columns="p_true")` (a real dataset doesn't come with the truth attached!), then:
- set `Age` to `NaN` for a random 20% of rows (`mask = rng.random(N) < 0.2`, then `data.loc[mask, "Age"] = np.nan`)
- set `Embarked` to `NaN` for a random 1% of rows

Keep `people` untouched. You'll need `p_true` later.

In [ ]:
data = ...
data.isnull().sum()

In [ ]:
assert "p_true" not in data.columns and "p_true" in people.columns
assert 0.17 < data["Age"].isnull().mean() < 0.23
assert 0 < data["Embarked"].isnull().sum() < 0.03 * N
assert people["Age"].notna().all(), "people should not be modified"
print("Exercise 11.4 passed")

### Exercise 11.5: Explore: does the data show the truth?
- `surv_table`: survival rate by `Sex` (rows) and `Pclass` (columns):
  `data.groupby(["Sex", "Pclass"])["Survived"].mean().unstack()`
- `max_gap`: the largest absolute difference between `surv_table` and the true **adult** rates in `P_SURVIVE`
  (build the truth as a DataFrame with the same shape, or compare row by row)

Why is the gap not exactly zero? Why would it be biggest for the *smallest* groups? (Hint: children, and sample size.)

In [ ]:
surv_table = ...
max_gap = ...
surv_table

In [ ]:
assert surv_table.shape == (2, 3)
assert surv_table.loc["female", 1] > surv_table.loc["male", 3]
assert 0 < max_gap < 0.12
print(f"Exercise 11.5 passed (max gap {max_gap:.3f})")

### Exercise 11.6: Split before you preprocess
Split into features and target, then into train and test sets:
- `X_df`: `data` without `Survived`, and `y`: the `Survived` column
- `X_train_df, X_test_df, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42, stratify=y)`

`stratify=y` keeps the survival rate the same in both parts. This is MLMA question 10: the split comes **first**, so the
test set can't influence anything we learn, not even a median.

In [ ]:
X_df = ...
y = ...
X_train_df, X_test_df, y_train, y_test = ..., ..., ..., ...

In [ ]:
assert len(X_train_df) == 0.8 * N and len(X_test_df) == 0.2 * N
assert "Survived" not in X_train_df.columns
assert abs(y_train.mean() - y_test.mean()) < 0.01
print("Exercise 11.6 passed")

### Exercise 11.7: Impute using training statistics only
- `age_median`: the median `Age` of the **training** set
- `port_mode`: the most common `Embarked` in the **training** set
- `train_clean`, `test_clean`: copies of `X_train_df` / `X_test_df` with both columns filled using those two values

In [ ]:
age_median = ...
port_mode = ...
train_clean = ...
test_clean = ...

In [ ]:
assert np.isclose(age_median, X_train_df["Age"].median())
assert train_clean.isnull().sum().sum() == 0 and test_clean.isnull().sum().sum() == 0
assert (test_clean.loc[X_test_df["Age"].isnull(), "Age"] == age_median).all(), "fill test ages with the TRAIN median"
assert X_train_df["Age"].isnull().any(), "don't modify X_train_df in place; work on copies"
print("Exercise 11.7 passed")

### Exercise 11.8: One-hot encoding with scikit-learn
1. `encoder = OneHotEncoder(drop="first", sparse_output=False)`, fitted on `train_clean[["Sex", "Embarked"]]`
2. build the numeric matrices `X_train` and `X_test` (NumPy arrays) whose columns are: `Pclass`, `Age`, then the encoded columns.
   Use `np.column_stack` or `np.hstack`.
3. `feature_names`: `["Pclass", "Age"] + list(encoder.get_feature_names_out(["Sex", "Embarked"]))`

In [ ]:
cat_cols = ["Sex", "Embarked"]
num_cols = ["Pclass", "Age"]
encoder = ...
X_train = ...
X_test = ...
feature_names = ...
feature_names

In [ ]:
assert X_train.shape == (len(train_clean), 5) and X_test.shape == (len(test_clean), 5)
assert feature_names == ["Pclass", "Age", "Sex_male", "Embarked_Q", "Embarked_S"]
assert np.array_equal(X_train[:, 2], (train_clean["Sex"] == "male").to_numpy().astype(float))
print("Exercise 11.8 passed")

### Exercise 11.9: Baseline vs. decision tree
- `baseline_acc`: accuracy on the test set of always predicting the most frequent class in `y_train`
- `clf`: a `DecisionTreeClassifier(max_depth=3, random_state=0)` fitted on the training data
- `tree_acc`: its accuracy on the test set (`accuracy_score`)

In [ ]:
baseline_acc = ...
clf = ...
tree_acc = ...
print(f"baseline {baseline_acc:.3f}, tree {tree_acc:.3f}")

In [ ]:
assert 0.55 < baseline_acc < 0.75
assert tree_acc > baseline_acc + 0.05, "the tree should clearly beat the baseline"
print("Exercise 11.9 passed")

### Exercise 11.10: The best possible accuracy
You know `p_true` for every person, so the **best any model could do** is to predict 1 exactly when `p_true > 0.5`.
That model would be right with probability `max(p, 1 - p)` for each person, so its expected accuracy on the test set is the
mean of `max(p_true, 1 - p_true)` over the test rows (this is the *Bayes-optimal* accuracy).

- `p_test`: `people.loc[X_test_df.index, "p_true"]`. The index connects rows across tables!
- `bayes_acc`: the expected best accuracy on the test set
- `gap`: `bayes_acc - tree_acc`

Is the tree close to the limit? Can *any* model reach 100% here? Why not?

In [ ]:
p_test = ...
bayes_acc = ...
gap = ...
print(f"tree {tree_acc:.3f}, best possible ~{bayes_acc:.3f}, gap {gap:.3f}")

In [ ]:
assert len(p_test) == len(y_test)
assert 0.70 < bayes_acc < 0.85
assert abs(gap) < 0.05, "a depth-3 tree should get within a few points of optimal"
print("Exercise 11.10 passed")

### Exercise 11.11: Predicted vs. true probabilities
`clf.predict_proba(X_test)` returns, for each row, `[P(class 0), P(class 1)]`. These are just the survival frequencies in the
tree leaf that the row lands in.
- `p_hat`: the predicted probability of survival (column 1)
- `mae`: the mean absolute difference between `p_hat` and `p_test`
- make a scatter plot of `p_hat` (y) against `p_test` (x) with `alpha=0.2`, plus the diagonal `ax.plot([0, 1], [0, 1], "--")`

Points on the diagonal mean the model recovered the truth exactly.

In [ ]:
p_hat = ...
mae = ...
plt.show()

In [ ]:
assert p_hat.shape == (len(y_test),) and ((0 <= p_hat) & (p_hat <= 1)).all()
assert mae < 0.12, f"mae {mae:.3f}"
print(f"Exercise 11.11 passed (mae {mae:.3f})")

### Exercise 11.12: How sure are we about the accuracy? A bootstrap
`tree_acc` comes from only 600 test passengers, so it is itself a random quantity. Estimate its uncertainty by **bootstrapping**:
repeat 2000 times: draw 600 test indices **with replacement** (`rng.integers(0, n_test, size=n_test)`) and compute the accuracy on that resample.

- `boot_accs`: array of the 2000 bootstrap accuracies (vectorize it: draw a `(2000, n_test)` index matrix and index the boolean array `correct` with it)
- `ci_low, ci_high`: the 2.5% and 97.5% percentiles (`np.percentile(boot_accs, [2.5, 97.5])`)

Does the interval contain `bayes_acc`? Plot a histogram of `boot_accs` with vertical lines at both ends of the interval.

In [ ]:
correct = (clf.predict(X_test) == y_test.to_numpy())
n_test = len(correct)
boot_rng = np.random.default_rng(0)
boot_accs = ...
ci_low, ci_high = ..., ...
plt.show()

In [ ]:
assert boot_accs.shape == (2000,)
assert ci_low < tree_acc < ci_high
assert 0.04 < ci_high - ci_low < 0.10, f"interval width {ci_high - ci_low:.3f}"
print("Exercise 11.12 passed")

### Bonus: look at the tree
This cell needs your `clf` and `feature_names` from Exercises 11.8–11.9.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(clf, filled=True, feature_names=feature_names, class_names=["Not Survived", "Survived"], rounded=True, ax=ax)
plt.show()

---
## You're done!

You've used every tool in MLMA Lab 1, and a few more that the probability labs rely on:

| tool | where you practised |
|---|---|
| core Python, functions, classes | Chapters 1–5 |
| NumPy arrays, broadcasting, `rng` simulation | Chapters 6–7, 11 |
| Matplotlib | Chapter 8 |
| `scipy.stats`, `quad`, `minimize`, MLE | Chapter 9 |
| pandas cleaning, grouping, conditional frequencies | Chapter 10 |
| scikit-learn pipeline, Bayes-optimal accuracy, bootstrap | Chapter 11 |

**Next step:** open `labs/2026_MLMA_Lab1.ipynb` and read it top to bottom. Every line should now be readable.
Then answer its 13 questions. Several of them (median vs. mean, mode for `Embarked`, splitting before preprocessing)
are things you've already done here.